# Figuring out how to automatically set up the metadata file needed for the process.py script

In [1]:
%cd ../..

/home/bhkuser/bhklab/katy/aaura-bench-preprocess


In [2]:
import pandas as pd
from pathlib import Path
from damply import dirs

In [3]:
dataset = "HCUCH_recist-dataset"
scan_dir_name = "images"
mask_dir_name = "masks"

In [ ]:
image_dir_path = dirs.RAWDATA / dataset / "images"

image_filepath_list = [i_path.relative_to(dirs.RAWDATA) for i_path in image_dir_path.rglob("*.nii.gz")]
image_filepath_list.sort()

In [ ]:
raw_aaura_metadata = {}

for idx, i_path in enumerate(image_filepath_list):
    raw_aaura_metadata[idx] = {
        'filepath': i_path.relative_to(dirs.RAWDATA),
        'source_id': f"{dataset}_{i_path.stem[0:3]}"
    }

    # rel_path = i_path.relative_to(dirs.RAWDATA)
    

In [12]:
raw_aaura_metadata_df = pd.DataFrame.from_dict(raw_aaura_metadata, orient='index')
raw_aaura_metadata_df.sort_values('source_id')

,filepath,source_id
28,HCUCH_recist-dataset/images/abdomen/train/imag...,HCUCH_recist-dataset_001
50,HCUCH_recist-dataset/images/abdomen/train/mask...,HCUCH_recist-dataset_001
100,HCUCH_recist-dataset/images/thorax/train/masks...,HCUCH_recist-dataset_001
84,HCUCH_recist-dataset/images/thorax/train/image...,HCUCH_recist-dataset_001
29,HCUCH_recist-dataset/images/abdomen/train/imag...,HCUCH_recist-dataset_002
...,...,...
48,HCUCH_recist-dataset/images/abdomen/train/imag...,HCUCH_recist-dataset_037
99,HCUCH_recist-dataset/images/thorax/train/image...,HCUCH_recist-dataset_037
115,HCUCH_recist-dataset/images/thorax/train/masks...,HCUCH_recist-dataset_037
71,HCUCH_recist-dataset/images/abdomen/train/mask...,HCUCH_recist-dataset_044


In [41]:
series_metadata_file = dirs.RAWDATA / dataset / "metadata" / "series.json"

series_metadata_df = pd.read_json(series_metadata_file, orient='records')
series_metadata_df.sort_values('patient_id', ignore_index=True, inplace=True)
series_metadata_df.head()

,id,region,uuid,study_uuid,study_date,patient_id,slice_thickness,row_spacing,column_spacing,slices,rows,columns,slice_spacing
0,369,abdomen,1.3.12.2.1107.5.1.4.83504.30000023042612315883...,1.3.51.0.1.1.172.19.3.128.3187796.3187735,20230426,1,5.0,0.808594,0.808594,195,512,512,2.5
1,371,thorax,1.3.12.2.1107.5.1.4.83504.30000023042612315883...,1.3.51.0.1.1.172.19.3.128.3187796.3187735,20230426,1,1.5,0.710938,0.710938,331,512,512,1.0
2,5,thorax,1.3.12.2.1107.5.1.4.83504.30000022071212080050...,1.3.51.0.1.1.172.19.3.128.3051489.3051428,20220712,2,1.5,0.601562,0.601562,324,512,512,1.0
3,6,abdomen,1.3.12.2.1107.5.1.4.83504.30000022071212080050...,1.3.51.0.1.1.172.19.3.128.3051489.3051428,20220712,2,5.0,0.736328,0.736328,179,512,512,2.5
4,612,abdomen,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,1.3.51.0.1.1.172.19.3.128.2857496.2857435,20210615,3,5.0,0.697266,0.697266,173,512,512,2.5


In [44]:
patient_metadata_file = dirs.RAWDATA / dataset / "metadata" / "patients.csv"
patient_metadata_df = pd.read_csv(patient_metadata_file)
patient_metadata_df.sort_values('patient_id', ignore_index=True, inplace=True)
patient_metadata_df.head()

,patient_id,subset,first_study_date,sex,age,diagnosis,histology,health_insurance
0,1,train,20230426,M,70,lung cancer,NaN,public
1,2,train,20220712,F,62,gastric cancer,NaN,uninsured
2,3,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,uninsured
3,5,train,20210527,F,59,colon cancer,Moderately Differentiated Adenocarcinoma,uninsured
4,7,train,20210526,M,53,rectal cancer,Well Differentiated Adenocarcinoma,uninsured


In [61]:
all_metadata_df = pd.merge(patient_metadata_df, series_metadata_df, how = "inner", left_on = 'patient_id', right_on = 'patient_id')
all_metadata_df.rename(columns={'patient_id': 'patient_number', 'uuid': 'series_uid'}, inplace=True)
all_metadata_df.insert(loc=0, column='filename', value=all_metadata_df.apply(lambda row: f"{row.patient_number:03d}_{row.series_uid}.nii.gz", axis=1))
all_metadata_df.insert(loc=1, column='patient_id', value=all_metadata_df.apply(lambda row: f"{dataset}_{row.patient_number:03d}", axis=1))
all_metadata_df.insert(loc=0, column = 'filepath', value=all_metadata_df.apply(lambda row: Path(dataset, 'images', row.region, row.subset, scan_dir_name, row.filename), axis=1))

In [62]:
all_metadata_df

,filepath,filename,patient_id,patient_number,subset,first_study_date,sex,age,diagnosis,histology,...,series_uid,study_uuid,study_date,slice_thickness,row_spacing,column_spacing,slices,rows,columns,slice_spacing
0,HCUCH_recist-dataset/images/abdomen/train/imag...,001_1.3.12.2.1107.5.1.4.83504.3000002304261231...,HCUCH_recist-dataset_001,1,train,20230426,M,70,lung cancer,NaN,...,1.3.12.2.1107.5.1.4.83504.30000023042612315883...,1.3.51.0.1.1.172.19.3.128.3187796.3187735,20230426,5.0,0.808594,0.808594,195,512,512,2.5
1,HCUCH_recist-dataset/images/thorax/train/image...,001_1.3.12.2.1107.5.1.4.83504.3000002304261231...,HCUCH_recist-dataset_001,1,train,20230426,M,70,lung cancer,NaN,...,1.3.12.2.1107.5.1.4.83504.30000023042612315883...,1.3.51.0.1.1.172.19.3.128.3187796.3187735,20230426,1.5,0.710938,0.710938,331,512,512,1.0
2,HCUCH_recist-dataset/images/thorax/train/image...,002_1.3.12.2.1107.5.1.4.83504.3000002207121208...,HCUCH_recist-dataset_002,2,train,20220712,F,62,gastric cancer,NaN,...,1.3.12.2.1107.5.1.4.83504.30000022071212080050...,1.3.51.0.1.1.172.19.3.128.3051489.3051428,20220712,1.5,0.601562,0.601562,324,512,512,1.0
3,HCUCH_recist-dataset/images/abdomen/train/imag...,002_1.3.12.2.1107.5.1.4.83504.3000002207121208...,HCUCH_recist-dataset_002,2,train,20220712,F,62,gastric cancer,NaN,...,1.3.12.2.1107.5.1.4.83504.30000022071212080050...,1.3.51.0.1.1.172.19.3.128.3051489.3051428,20220712,5.0,0.736328,0.736328,179,512,512,2.5
4,HCUCH_recist-dataset/images/abdomen/test/image...,003_1.3.12.2.1107.5.1.4.83504.3000002106150914...,HCUCH_recist-dataset_003,3,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,...,1.3.12.2.1107.5.1.4.83504.30000021061509140333...,1.3.51.0.1.1.172.19.3.128.2857496.2857435,20210615,5.0,0.697266,0.697266,173,512,512,2.5
5,HCUCH_recist-dataset/images/abdomen/test/image...,003_1.3.12.2.1107.5.1.4.83504.3000002108250932...,HCUCH_recist-dataset_003,3,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,...,1.3.12.2.1107.5.1.4.83504.30000021082509324422...,1.3.51.0.1.1.172.19.3.128.2890383.2890322,20210825,5.0,0.677734,0.677734,175,512,512,2.5
6,HCUCH_recist-dataset/images/abdomen/test/image...,003_1.3.12.2.1107.5.1.4.83504.3000002110271233...,HCUCH_recist-dataset_003,3,test,20210615,F,61,gallbladder cancer,Poorly Differentiated Adenocarcinoma with Sign...,...,1.3.12.2.1107.5.1.4.83504.30000021102712332815...,1.3.51.0.1.1.172.19.3.128.2891111.2891050,20211027,5.0,0.628906,0.628906,177,512,512,2.5
7,HCUCH_recist-dataset/images/abdomen/train/imag...,005_1.3.12.2.1107.5.1.4.83504.3000002105271609...,HCUCH_recist-dataset_005,5,train,20210527,F,59,colon cancer,Moderately Differentiated Adenocarcinoma,...,1.3.12.2.1107.5.1.4.83504.30000021052716090067...,1.3.51.0.1.1.172.19.3.128.2847403.2847342,20210527,5.0,0.710938,0.710938,181,512,512,2.5
8,HCUCH_recist-dataset/images/abdomen/train/imag...,007_1.3.12.2.1107.5.1.4.83504.3000002110071227...,HCUCH_recist-dataset_007,7,train,20210526,M,53,rectal cancer,Well Differentiated Adenocarcinoma,...,1.3.12.2.1107.5.1.4.83504.30000021100712273932...,1.3.51.0.1.1.172.19.3.128.2890368.2890307,20211007,5.0,0.703125,0.703125,192,512,512,2.5
9,HCUCH_recist-dataset/images/abdomen/train/imag...,007_1.3.12.2.1107.5.1.4.83504.3000002108021212...,HCUCH_recist-dataset_007,7,train,20210526,M,53,rectal cancer,Well Differentiated Adenocarcinoma,...,1.3.12.2.1107.5.1.4.83504.30000021080212120090...,1.3.51.0.1.1.172.19.3.128.2879716.2879655,20210802,5.0,0.781250,0.781250,218,512,512,2.5


# Steps for preprocess metadata generation  
1. Load in any metadata files provided  
    a. merge everything
1. Make patient_id column with dataset and patient number
1. User specifies how to construct the filename and the filepath from columns in the metadata or other variables  
    a. Like scan directory name vs. mask directory name, since niftis are usually sorted by "images" and "masks" or "labels"
1. Make the filename column to be used to find the image and mask file



In [3]:
from typing import Optional, ClassVar
from pydantic import BaseModel, Field, field_validator, ValidationInfo

class NiftiDatasetConfig(BaseModel):
    """Configuration for nifti image dataset to create aaura index for."""

    datasource: str = Field(..., description="The source of the dataset (e.g. TCIA, PMCC)")
    dataset: str = Field(..., description="The name of the dataset (e.g. RADCURE)")
    patient_id_col: str = Field(default="patient_id", description="The name of the column in the metadata files that contains the patient ID")
    scan_name_pattern: str = Field(default="{patient_id}.nii.gz", description="The pattern to use for generating filenames in the index. Use {patient_id} and {series_uid} as placeholders for the patient ID and series UID, respectively.")
    mask_name_pattern : Optional[str] = Field(default=None, validate_default=True,description="The pattern to use for generating mask filenames in the index. Use {patient_id} and {series_uid} as placeholders for the patient ID and series UID, respectively.")
    scan_path_pattern: Optional[str] = Field(default="scans", description="The pattern to use for generating filepaths to the scan images starting after {datasource}_{dataset}/images.")
    mask_path_pattern: Optional[str] = Field(default="masks", description="The pattern to use for generating filepaths to the segmentation masks starting after {datasource}_{dataset}/images.")
    disease_site: Optional[str] = Field(default=None, description="The disease site of the dataset (e.g. lung)")
    metadata_files: Optional[list[str]] = Field(default=[], description="List of paths to metadata files to include in the index") 
    image_modality: Optional[str] = Field(default="CT", description="The modality of the images (e.g. CT)")
    drop_data: Optional[dict[str, list[str]]] = Field(default=None, description="Dictionary specifying any data to drop from the index (e.g. {'patient_id': ['TCGA-02-0047']})")
    anatomy_match_file: Optional[str] = Field(default=None, description="Path to a separate metadata file containing mappings of sources to anatomy labels, for use when dataset contains multiple other datasets with different disease sites.")

    @field_validator('scan_name_pattern')
    def validate_filename_pattern(cls, v):
        if ".nii.gz" not in v:
            return f"{v}.nii.gz"
        return v
    
    @field_validator('mask_name_pattern')
    def validate_maskname_pattern(cls, v, info: ValidationInfo):
        if v is not None and ".nii.gz" not in v:
            return f"{v}.nii.gz"
        elif v is None:
            v = info.data['scan_name_pattern']
        return v

    @field_validator('metadata_files')
    @classmethod
    def validate_metadata_files(cls, v: list[str], info: ValidationInfo):
        if not isinstance(v, list):
            raise ValueError("metadata_files must be a list")
        elif len(v) == 0:
            print("No metadata files specified. Attempting to find metadata files in expected location...")
            metadata_dir_path = dirs.RAWDATA / f"{info.data['datasource']}_{info.data['dataset']}/metadata"
            v = list(metadata_dir_path.glob(r"*.csv")) + list(metadata_dir_path.glob(r"*.json"))
            if len(v) == 0:
                raise ValueError(f"No metadata files found in expected location: {metadata_dir_path}")
            if info.data['anatomy_match_file'] is not None:
                v.drop(info.data['anatomy_match_file'], inplace=True)
        return v

In [5]:
recist_data_config = NiftiDatasetConfig(
    datasource="HCUCH",
    dataset="recist-dataset",
    scan_name_pattern="{patient_id:03d}_{series_uid}.nii.gz",
    scan_path_pattern="{region}/{subset}/images",
    mask_path_pattern="{region}/{subset}/masks",
    metadata_files=['patients.csv', 'series.json'],
    image_modality="CT")

recist_data_config

NiftiDatasetConfig(datasource='HCUCH', dataset='recist-dataset', patient_id_col='patient_id', scan_name_pattern='{patient_id:03d}_{series_uid}.nii.gz', mask_name_pattern='{patient_id:03d}_{series_uid}.nii.gz', scan_path_pattern='{region}/{subset}/images', mask_path_pattern='{region}/{subset}/masks', disease_site=None, metadata_files=['patients.csv', 'series.json'], image_modality='CT', drop_data=None, anatomy_match_file=None)

In [6]:
lesion_locator_config = NiftiDatasetConfig(
    datasource = "CVPR",
    dataset = "LesionLocator",
    patient_id_col = "File Name",
    scan_name_pattern="{patient_id}_0000.nii.gz",
    mask_name_pattern="{patient_id}.nii.gz",
    scan_path_pattern="images/{timepoint}/images",
    mask_path_pattern="masks/{timepoint}/labels",
    metadata_files = ["10_example_cases.csv"],
    image_modality = "CT",
    drop_data = {"Source": ['coronacases','NIH-LYMPH']},
    anatomy_match_file="dataset_anatomy_match.csv"
)

lesion_locator_config

NiftiDatasetConfig(datasource='CVPR', dataset='LesionLocator', patient_id_col='File Name', scan_name_pattern='{patient_id}_0000.nii.gz', mask_name_pattern='{patient_id}.nii.gz', scan_path_pattern='images/{timepoint}/images', mask_path_pattern='masks/{timepoint}/labels', disease_site=None, metadata_files=['10_example_cases.csv'], image_modality='CT', drop_data={'Source': ['coronacases', 'NIH-LYMPH']}, anatomy_match_file='dataset_anatomy_match.csv')

In [23]:
mama_mia_config = NiftiDatasetConfig(
    datasource='BCN-AIM',
    dataset='MAMA-MIA',
    patient_id_col = 'patient_id',
    scan_name_pattern="{patient_id}_0000.nii.gz",
    mask_name_pattern="{patient_id}.nii.gz",
    metadata_files = ['clinical_and_imaging_info.csv'],
    scan_path_pattern="scans/{patient_id}",
    mask_path_pattern="masks",
    image_modality="MR",
    disease_site='breast'
)

mama_mia_config

NiftiDatasetConfig(datasource='BCN-AIM', dataset='MAMA-MIA', patient_id_col='patient_id', scan_name_pattern='{patient_id}_0000.nii.gz', mask_name_pattern='{patient_id}.nii.gz', scan_path_pattern='scans/{patient_id}', mask_path_pattern='masks', disease_site='breast', metadata_files=['clinical_and_imaging_info.csv'], image_modality='MR', drop_data=None, anatomy_match_file=None)

In [24]:
config = mama_mia_config

metadata_df = pd.read_csv(dirs.RAWDATA / f"{config.datasource}_{config.dataset}" / "metadata" / config.metadata_files[0])

In [25]:
from imgtools.pattern_parser import PatternResolver

def image_path_resolver(metadata_row:pd.Series,
                        image_path_pattern:str,
                        image_file_pattern:str) -> str:
    
    filename_format = f"{image_path_pattern}/{image_file_pattern}"

    image_path_resolver = PatternResolver(filename_format)

    return image_path_resolver.resolve(metadata_row.to_dict())

In [26]:
metadata_df.apply(lambda row: image_path_resolver(row, config.scan_path_pattern, config.scan_name_pattern), axis=1)

0       scans/DUKE_001/DUKE_001_0000.nii.gz
1       scans/DUKE_002/DUKE_002_0000.nii.gz
2       scans/DUKE_005/DUKE_005_0000.nii.gz
3       scans/DUKE_009/DUKE_009_0000.nii.gz
4       scans/DUKE_010/DUKE_010_0000.nii.gz
                       ...                 
1501      scans/NACT_64/NACT_64_0000.nii.gz
1502      scans/NACT_65/NACT_65_0000.nii.gz
1503      scans/NACT_66/NACT_66_0000.nii.gz
1504      scans/NACT_67/NACT_67_0000.nii.gz
1505      scans/NACT_68/NACT_68_0000.nii.gz
Length: 1506, dtype: object

In [27]:
metadata_df.apply(
        lambda row: Path(f"{config.datasource}_{config.dataset}") / "images" / image_path_resolver(row, config.mask_path_pattern, config.mask_name_pattern), 
        axis = 1
        )

0       BCN-AIM_MAMA-MIA/images/masks/DUKE_001.nii.gz
1       BCN-AIM_MAMA-MIA/images/masks/DUKE_002.nii.gz
2       BCN-AIM_MAMA-MIA/images/masks/DUKE_005.nii.gz
3       BCN-AIM_MAMA-MIA/images/masks/DUKE_009.nii.gz
4       BCN-AIM_MAMA-MIA/images/masks/DUKE_010.nii.gz
                            ...                      
1501     BCN-AIM_MAMA-MIA/images/masks/NACT_64.nii.gz
1502     BCN-AIM_MAMA-MIA/images/masks/NACT_65.nii.gz
1503     BCN-AIM_MAMA-MIA/images/masks/NACT_66.nii.gz
1504     BCN-AIM_MAMA-MIA/images/masks/NACT_67.nii.gz
1505     BCN-AIM_MAMA-MIA/images/masks/NACT_68.nii.gz
Length: 1506, dtype: object

In [ ]:
from imgtools.pattern_parser import PatternResolver

pattern = config.scan_path_pattern

for sample_tuple in metadata_df.itertuples(index=False):

    scan_path_resolver = PatternResolver(filename_format=config.scan_path_pattern)
    scan_path = scan_path_resolver.resolve(sample_tuple._asdict())
    print(scan_path)

scans/DUKE_001
scans/DUKE_002
scans/DUKE_005
scans/DUKE_009
scans/DUKE_010
scans/DUKE_012
scans/DUKE_019
scans/DUKE_021
scans/DUKE_022
scans/DUKE_028
scans/DUKE_032
scans/DUKE_034
scans/DUKE_040
scans/DUKE_041
scans/DUKE_043
scans/DUKE_044
scans/DUKE_045
scans/DUKE_046
scans/DUKE_048
scans/DUKE_051
scans/DUKE_055
scans/DUKE_057
scans/DUKE_059
scans/DUKE_060
scans/DUKE_061
scans/DUKE_064
scans/DUKE_069
scans/DUKE_071
scans/DUKE_077
scans/DUKE_082
scans/DUKE_086
scans/DUKE_090
scans/DUKE_091
scans/DUKE_097
scans/DUKE_099
scans/DUKE_101
scans/DUKE_103
scans/DUKE_104
scans/DUKE_105
scans/DUKE_107
scans/DUKE_114
scans/DUKE_115
scans/DUKE_116
scans/DUKE_117
scans/DUKE_119
scans/DUKE_120
scans/DUKE_123
scans/DUKE_129
scans/DUKE_132
scans/DUKE_134
scans/DUKE_136
scans/DUKE_137
scans/DUKE_141
scans/DUKE_142
scans/DUKE_144
scans/DUKE_148
scans/DUKE_150
scans/DUKE_156
scans/DUKE_157
scans/DUKE_160
scans/DUKE_163
scans/DUKE_167
scans/DUKE_168
scans/DUKE_176
scans/DUKE_177
scans/DUKE_178
scans/DUKE

In [45]:
scan_path_pattern = config.scan_path_pattern

# [id for id in metadata_df['patient_id'].to_list()]

metadata_df['patient_id'].apply(lambda patient_id: Path(f"{config.datasource}_{config.dataset}") / "images" / scan_path_pattern.format(patient_id=patient_id))

# [scan_path_pattern.format(id) for id in 
# metadata_df['patient_id']

0       BCN-AIM_MAMA-MIA/images/scans/DUKE_001
1       BCN-AIM_MAMA-MIA/images/scans/DUKE_002
2       BCN-AIM_MAMA-MIA/images/scans/DUKE_005
3       BCN-AIM_MAMA-MIA/images/scans/DUKE_009
4       BCN-AIM_MAMA-MIA/images/scans/DUKE_010
                         ...                  
1501     BCN-AIM_MAMA-MIA/images/scans/NACT_64
1502     BCN-AIM_MAMA-MIA/images/scans/NACT_65
1503     BCN-AIM_MAMA-MIA/images/scans/NACT_66
1504     BCN-AIM_MAMA-MIA/images/scans/NACT_67
1505     BCN-AIM_MAMA-MIA/images/scans/NACT_68
Name: patient_id, Length: 1506, dtype: object

In [25]:
scan_path = config.scan_path_pattern

for idx, row in metadata_df.iterrows():
    patient_id = row.patient_id
    print(scan_path.format(patient_id=patient_id))

scans/DUKE_001
scans/DUKE_002
scans/DUKE_005
scans/DUKE_009
scans/DUKE_010
scans/DUKE_012
scans/DUKE_019
scans/DUKE_021
scans/DUKE_022
scans/DUKE_028
scans/DUKE_032
scans/DUKE_034
scans/DUKE_040
scans/DUKE_041
scans/DUKE_043
scans/DUKE_044
scans/DUKE_045
scans/DUKE_046
scans/DUKE_048
scans/DUKE_051
scans/DUKE_055
scans/DUKE_057
scans/DUKE_059
scans/DUKE_060
scans/DUKE_061
scans/DUKE_064
scans/DUKE_069
scans/DUKE_071
scans/DUKE_077
scans/DUKE_082
scans/DUKE_086
scans/DUKE_090
scans/DUKE_091
scans/DUKE_097
scans/DUKE_099
scans/DUKE_101
scans/DUKE_103
scans/DUKE_104
scans/DUKE_105
scans/DUKE_107
scans/DUKE_114
scans/DUKE_115
scans/DUKE_116
scans/DUKE_117
scans/DUKE_119
scans/DUKE_120
scans/DUKE_123
scans/DUKE_129
scans/DUKE_132
scans/DUKE_134
scans/DUKE_136
scans/DUKE_137
scans/DUKE_141
scans/DUKE_142
scans/DUKE_144
scans/DUKE_148
scans/DUKE_150
scans/DUKE_156
scans/DUKE_157
scans/DUKE_160
scans/DUKE_163
scans/DUKE_167
scans/DUKE_168
scans/DUKE_176
scans/DUKE_177
scans/DUKE_178
scans/DUKE

In [ ]:
metadata_df.patient_id.apply(lambda row: f"{config.scan_path_pattern}")

0       scans/{patient_id}
1       scans/{patient_id}
2       scans/{patient_id}
3       scans/{patient_id}
4       scans/{patient_id}
               ...        
1501    scans/{patient_id}
1502    scans/{patient_id}
1503    scans/{patient_id}
1504    scans/{patient_id}
1505    scans/{patient_id}
Name: patient_id, Length: 1506, dtype: object

In [ ]:
metadata_df

,patient_id,dataset,bilateral_breast_cancer,multifocal_cancer,nac_agent,endocrine_therapy,anti_her2_neu_therapy,pcr,mastectomy_post_nac,days_to_follow_up,...,scanner_model,high_bit,window_center,window_width,echo_time,repetition_time,acquisition_times,acquisition_date,tcia_series_uid,raw_scan_path
0,DUKE_001,DUKE,0,0.0,NaN,0.0,1.0,0.0,0.0,2940.0,...,Avanto,11,53.0,145.0,1.360,4.120,"[0, 584, 714, 846, 977]",NaN,1.3.6.1.4.1.14519.5.2.1.1857778498036652445367...,NaN
1,DUKE_002,DUKE,0,0.0,NaN,0.0,0.0,0.0,0.0,1649.0,...,Signa HDxt,15,326.0,652.0,2.704,6.918,"[0, 165, 288, 411]",NaN,1.3.6.1.4.1.14519.5.2.1.2920879504444133146439...,NaN
2,DUKE_005,DUKE,0,1.0,NaN,0.0,1.0,1.0,1.0,1845.0,...,Avanto,11,106.0,288.0,1.340,4.270,"[0, 183, 289, 398]",NaN,1.3.6.1.4.1.14519.5.2.1.3082672827908226352755...,NaN
3,DUKE_009,DUKE,0,1.0,NaN,0.0,0.0,1.0,0.0,1554.0,...,Signa HDxt,15,231.0,462.0,2.604,6.032,"[0, 199, 314, 429]",NaN,1.3.6.1.4.1.14519.5.2.1.1955995935901349215652...,NaN
4,DUKE_010,DUKE,0,1.0,NaN,0.0,0.0,0.0,1.0,384.0,...,Signa HDxt,15,532.0,1064.0,2.436,5.668,"[0, 156, 255, 355]",NaN,1.3.6.1.4.1.14519.5.2.1.1390661710325525999628...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,NACT_64,NACT,0,0.0,FEC100,NaN,NaN,0.0,1.0,705.0,...,GENESIS_SIGNA,15,15.0,31.0,4.200,8.400,"[0, 403, 751]",1994/02/23,1.3.6.1.4.1.14519.5.2.1.7695.2311.188949018082...,NaN
1502,NACT_65,NACT,0,0.0,Anthracycline,NaN,NaN,0.0,1.0,1448.0,...,GENESIS_SIGNA,15,34.0,68.0,4.200,8.000,"[0, 380, 686]",1992/06/16,1.3.6.1.4.1.14519.5.2.1.7695.2311.196765391224...,NaN
1503,NACT_66,NACT,0,0.0,Anthracycline,NaN,NaN,0.0,0.0,667.0,...,GENESIS_SIGNA,15,32.0,64.0,4.200,8.000,"[0, 362, 678, 962]",1992/01/07,1.3.6.1.4.1.14519.5.2.1.7695.2311.757338311647...,NaN
1504,NACT_67,NACT,0,0.0,Anthracycline,NaN,NaN,0.0,1.0,1234.0,...,GENESIS_SIGNA,15,NaN,NaN,4.200,11.000,"[0, 441, 947]",1988/04/21,1.3.6.1.4.1.14519.5.2.1.7695.2311.298593028606...,NaN
